💡 **Environment:** `clamp-analyses`

# Description

**Sandbox NB 01/03 — null-distribution adjustment of the lipid-disease scores.**

For each method (single-gene, module-based ARCHS4) and each lipid DOID, computes three
adjusted scores against the raw `-drugᵀ·disease` score from NB00:

1. **Permute-disease (primary).** Permute the disease gene/LV vector `B=1000` times (seeded),
   recompute scores → per-drug null. Report `NES = (s-μ)/σ` and one-sided `p = (1+#{null≥obs})/(B+1)`.
2. **Analytic Pearson.** `adj_pearson = -corr(drug, disease)`. The permutation NES is proportional
   to this; we **assert** `corr(NES_permute, adj_pearson) > 0.95` at the trait level (sanity gate).
3. **Background-trait null.** For each drug, compare its score for the lipid DOID to its scores
   across all *other* mapped DOIDs → `NES_bg`, `p_bg`. Preserves gene-gene correlation.

Per-trait adjusted scores are aggregated to the DOID by `max` over the contributing UKB trait(s),
mirroring `map_traits_to_doid`. See `CLAUDE.md`.

# Modules loading

In [1]:
import json
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [2]:
SEED = 42
N_PERM = 1000

LIPID_DOIDS = ['DOID:1936', 'DOID:3393']
METHODS = ['gene_based', 'module_based_archs4']

DATA_DIR = here('data/drug_disease_associations')
OUTPUT_DIR = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/null_adjust_test')
assert OUTPUT_DIR.exists(), 'run NB00 first'

LINCS_RAW_FILE = DATA_DIR / 'lincs-data.pkl'
SPREDIXCAN_RAW_LIVER = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    '00_spredixcan_projection_archs4/spredixcan/raw/'
    'spredixcan-mashr-zscores-Liver-data.pkl')
LINCS_PROJ_FILE = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    '01_lincs_projection_archs4/lincs/lincs-projection.pkl')
SPREDIXCAN_PROJ_LIVER = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    '00_spredixcan_projection_archs4/spredixcan/proj/'
    'spredixcan-mashr-zscores-Liver-projection-archs4.pkl')

# Load NB00 outputs + the source matrices

In [3]:
observed = pd.read_pickle(OUTPUT_DIR / 'observed_lipid_scores.pkl')
scores_doid = pd.read_pickle(OUTPUT_DIR / 'observed_doid_matrices.pkl')  # {method: drugs x DOID}
contrib_traits = json.loads((OUTPUT_DIR / 'contrib_traits.json').read_text())
display(contrib_traits)

# Reload the drug (L) and disease (D) matrices, same as NB00
L = {'gene_based': pd.read_pickle(LINCS_RAW_FILE),
     'module_based_archs4': pd.read_pickle(LINCS_PROJ_FILE)}
D = {'gene_based': pd.read_pickle(SPREDIXCAN_RAW_LIVER),
     'module_based_archs4': pd.read_pickle(SPREDIXCAN_PROJ_LIVER)}

# Per-method common index (genes or LVs) and aligned numpy drug matrix
COMMON, LMAT, DRUGS = {}, {}, {}
for m in METHODS:
    common = L[m].index.intersection(D[m].index)
    COMMON[m] = common
    LMAT[m] = L[m].loc[common].values          # (k x drugs)
    DRUGS[m] = list(L[m].columns)
    print(m, '| k =', len(common), '| drugs =', len(DRUGS[m]))

{'DOID:1936': ['I70_Diagnoses_main_ICD10_I70_Atherosclerosis'],
 'DOID:3393': ['20002_1075_Noncancer_illness_code_selfreported_heart_attackmyocardial_infarction',
  'CARDIoGRAM_C4D_CAD_ADDITIVE',
  'I25_Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease']}

gene_based | k = 5182 | drugs = 1170
module_based_archs4 | k = 1728 | drugs = 1170


# Core null / adjustment routines (trait level)

In [4]:
def trait_vector(method, trait):
    """Disease vector (k,) aligned to the method's common gene/LV index."""
    return D[method].loc[COMMON[method], trait].values


def observed_trait_scores(method, x):
    """Raw pipeline score per drug for one disease vector: s = -Lᵀ x."""
    return -(LMAT[method].T @ x)                # (drugs,)


def permute_null(method, x, n_perm=N_PERM, seed=SEED):
    """Permute the disease vector n_perm times -> per-drug NES and one-sided p."""
    rng = np.random.default_rng(seed)
    k = x.shape[0]
    idx = np.argsort(rng.random((n_perm, k)), axis=1)   # n_perm independent permutations
    Xp = x[idx].T                                        # (k x n_perm)
    null = -(LMAT[method].T @ Xp)                        # (drugs x n_perm)
    s_obs = observed_trait_scores(method, x)             # (drugs,)
    mu = null.mean(axis=1)
    sd = null.std(axis=1, ddof=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        nes = (s_obs - mu) / sd
    p = (1.0 + (null >= s_obs[:, None]).sum(axis=1)) / (n_perm + 1.0)
    return s_obs, nes, p


def pearson_adj(method, x):
    """Analytic equivalent: adj_pearson = -corr(drug_signature, disease_vector), per drug."""
    Lc = LMAT[method] - LMAT[method].mean(axis=0, keepdims=True)   # center each drug column
    xc = x - x.mean()
    num = Lc.T @ xc
    den = np.sqrt((Lc ** 2).sum(axis=0) * (xc ** 2).sum())
    with np.errstate(divide='ignore', invalid='ignore'):
        r = num / den
    return -r

# Compute trait-level adjustments + sanity gate (NES_permute ≈ analytic Pearson)

In [5]:
trait_records = {}   # (method, trait) -> dict of per-drug arrays
gate = []
for m in METHODS:
    for d in LIPID_DOIDS:
        for t in contrib_traits[d]:
            x = trait_vector(m, t)
            s_obs, nes, p = permute_null(m, x)
            pear = pearson_adj(m, x)
            trait_records[(m, t)] = dict(drug=DRUGS[m], s_obs=s_obs, nes=nes, p=p, pearson=pear)
            ok = np.isfinite(nes) & np.isfinite(pear)
            r = np.corrcoef(nes[ok], pear[ok])[0, 1]
            gate.append({'method': m, 'doid': d, 'trait': t, 'corr_nes_pearson': r})

gate = pd.DataFrame(gate)
display(gate)
assert (gate['corr_nes_pearson'] > 0.95).all(), \
    'permutation NES diverges from analytic Pearson -- check the null'
print('SANITY GATE PASSED: NES_permute is proportional to analytic Pearson (corr > 0.95).')

,method,doid,trait,corr_nes_pearson
0,gene_based,DOID:1936,I70_Diagnoses_main_ICD10_I70_Atherosclerosis,0.999326
1,gene_based,DOID:3393,20002_1075_Noncancer_illness_code_selfreported...,0.999373
2,gene_based,DOID:3393,CARDIoGRAM_C4D_CAD_ADDITIVE,0.999124
3,gene_based,DOID:3393,I25_Diagnoses_main_ICD10_I25_Chronic_ischaemic...,0.999324
4,module_based_archs4,DOID:1936,I70_Diagnoses_main_ICD10_I70_Atherosclerosis,0.999273
5,module_based_archs4,DOID:3393,20002_1075_Noncancer_illness_code_selfreported...,0.999502
6,module_based_archs4,DOID:3393,CARDIoGRAM_C4D_CAD_ADDITIVE,0.999285
7,module_based_archs4,DOID:3393,I25_Diagnoses_main_ICD10_I25_Chronic_ischaemic...,0.999465


SANITY GATE PASSED: NES_permute is proportional to analytic Pearson (corr > 0.95).


# Aggregate trait → DOID by `max` (mirrors map_traits_to_doid)

The DOID-level adjusted score is the max over contributing traits of the per-trait adjusted score
(NES / Pearson); the reported p is the one at the trait achieving that max.

In [6]:
def aggregate_doid(method, doid):
    traits = contrib_traits[doid]
    drugs = DRUGS[method]
    nes = np.vstack([trait_records[(method, t)]['nes'] for t in traits])       # (T x drugs)
    p = np.vstack([trait_records[(method, t)]['p'] for t in traits])
    pear = np.vstack([trait_records[(method, t)]['pearson'] for t in traits])
    s_obs = np.vstack([trait_records[(method, t)]['s_obs'] for t in traits])
    amax = np.nanargmax(nes, axis=0)                                           # winning trait / drug
    cols = np.arange(len(drugs))
    return pd.DataFrame({
        'method': method, 'doid': doid, 'drug': drugs,
        'raw_score': s_obs.max(axis=0),               # == NB00 raw DOID score (asserted below)
        'nes_permute': nes[amax, cols],
        'p_permute': p[amax, cols],
        'pearson': np.nanmax(pear, axis=0),
    })

adj = pd.concat([aggregate_doid(m, d) for m in METHODS for d in LIPID_DOIDS], ignore_index=True)

# sanity: raw_score here reproduces NB00's DOID-level matrix
for m in METHODS:
    for d in LIPID_DOIDS:
        a = adj[(adj.method == m) & (adj.doid == d)].set_index('drug')['raw_score']
        ref = scores_doid[m][d].reindex(a.index)
        assert np.allclose(a.values, ref.values, atol=1e-6), (m, d)
print('raw_score reproduces NB00 DOID matrix.')

raw_score reproduces NB00 DOID matrix.


# Background-trait null (DOID level)

In [7]:
# For each drug, null = its scores across all OTHER mapped DOIDs (exclude the lipid DOIDs).
def background_adjust(method, doid):
    mat = scores_doid[method]                                   # drugs x DOID
    bg_cols = [c for c in mat.columns if c not in LIPID_DOIDS]
    bg = mat[bg_cols].values                                    # drugs x N_bg
    obs = mat[doid].values                                      # drugs
    mu = bg.mean(axis=1)
    sd = bg.std(axis=1, ddof=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        nes_bg = (obs - mu) / sd
    p_bg = (1.0 + (bg >= obs[:, None]).sum(axis=1)) / (bg.shape[1] + 1.0)
    return pd.DataFrame({'method': method, 'doid': doid, 'drug': list(mat.index),
                         'nes_bg': nes_bg, 'p_bg': p_bg})

bg = pd.concat([background_adjust(m, d) for m in METHODS for d in LIPID_DOIDS], ignore_index=True)
adj = adj.merge(bg, on=['method', 'doid', 'drug'], how='left')

# Attach gold-standard labels + readable names; save

In [8]:
obs_small = observed[['method', 'doid', 'drug', 'drug_name', 'true_class']]
adj = adj.merge(obs_small, on=['method', 'doid', 'drug'], how='left')

# rank columns (1 = best) for each score variant, within method x DOID
def add_rank(df, col):
    df[col + '_rank'] = (df.groupby(['method', 'doid'])[col]
                         .rank(ascending=False, method='min').astype(int))
for col in ['raw_score', 'nes_permute', 'pearson', 'nes_bg']:
    add_rank(adj, col)

adj.to_pickle(OUTPUT_DIR / 'adjusted_lipid_scores.pkl')
gate.to_csv(OUTPUT_DIR / 'pearson_sanity_gate.csv', index=False)
print('adjusted_lipid_scores.pkl:', adj.shape)
display(adj.head())

adjusted_lipid_scores.pkl: (4680, 15)


,method,doid,drug,raw_score,nes_permute,p_permute,pearson,nes_bg,p_bg,drug_name,true_class,raw_score_rank,nes_permute_rank,pearson_rank,nes_bg_rank
0,gene_based,DOID:1936,DB00014,1.900449,0.059239,0.484515,0.000696,-0.128866,0.584022,Goserelin,NaN,597,586,589,574
1,gene_based,DOID:1936,DB00091,-117.694398,-0.467339,0.697303,-0.006382,-0.595319,0.738292,Cyclosporine,NaN,1101,811,802,773
2,gene_based,DOID:1936,DB00121,52.053343,0.668276,0.245754,0.009504,0.512779,0.300275,Biotin,NaN,293,314,310,286
3,gene_based,DOID:1936,DB00130,-3.387907,-0.000656,0.505495,-0.000609,-0.225693,0.606061,L-Glutamine,NaN,631,613,630,614
4,gene_based,DOID:1936,DB00131,10.033468,0.142466,0.452547,0.002191,-0.043313,0.539945,Adenosine monophosphate,NaN,538,548,541,537


# Save showcase null distributions (for NB02 histograms)

In [9]:
# A few (method, doid, drug) cells: a statin and a known negative, for the single-trait DOID:1936.
SHOWCASE = [
    ('gene_based', 'DOID:1936', 'DB00641'),         # Simvastatin (positive)
    ('module_based_archs4', 'DOID:1936', 'DB00641'),
    ('gene_based', 'DOID:1936', 'DB00758'),         # Clopidogrel (negative)
]
rng_seed = SEED
showcase = {}
for m, d, drug in SHOWCASE:
    t = contrib_traits[d][0]                          # DOID:1936 has a single trait
    x = trait_vector(m, t)
    rng = np.random.default_rng(rng_seed)
    k = x.shape[0]
    idx = np.argsort(rng.random((N_PERM, k)), axis=1)
    null = -(LMAT[m].T @ x[idx].T)                    # (drugs x N_PERM)
    di = DRUGS[m].index(drug)
    showcase[(m, d, drug)] = {
        'null': null[di], 'obs': observed_trait_scores(m, x)[di],
        'drug_name': observed.query('method==@m and doid==@d and drug==@drug')['drug_name'].iloc[0],
    }
pd.to_pickle(showcase, OUTPUT_DIR / 'showcase_null.pkl')
print('saved showcase_null.pkl for', list(showcase.keys()))

saved showcase_null.pkl for [('gene_based', 'DOID:1936', 'DB00641'), ('module_based_archs4', 'DOID:1936', 'DB00641'), ('gene_based', 'DOID:1936', 'DB00758')]
